# Part 2 - Classification

> Goal: Implement core classification models from scratch, evaluate with robust metrics/tests, and compare with sklearn baselines.

## Reproducibility Checklist
- Set random seed globally and per algorithm.
- Keep experiment logs for hyperparameters, class weighting, and metrics.
- Separate from-scratch implementation and sklearn benchmark in different cells/sections.

In [1]:
# Setup
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Optional sklearn baselines for comparison only
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold

# Local utils
from utils import set_seed, load_dataset, describe_dataset, train_val_test_split, log_experiment_params

SEED = 42
set_seed(SEED)
np.set_printoptions(suppress=True, precision=4)

ROOT = Path.cwd()
DATA_PATH = ROOT.parent.parent / "data" / "raw" / "classification" / "covtype.csv"

print(f"Using dataset: {DATA_PATH}")

Using dataset: d:\Ki2Nam3\Machine Learning\Projects\Group_14\data\raw\classification\covtype.csv


## 1. Introduction
- Task definition
- Dataset overview

















In [2]:
# 1) Load + describe dataset
df = load_dataset(str(DATA_PATH))
summary = describe_dataset(df)
print('Shape:', summary['shape'])

# TODO: define target column name
TARGET_COL = 'Cover_Type'  # adjust if needed
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# 2) Split data
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(
    X, y, train_ratio=0.7, val_ratio=0.1, seed=SEED
)

Shape: (581012, 55)


## 2. EDA
- class distribution
- imbalance analysis


In [3]:
# 3) EDA placeholders
# TODO: class distribution and imbalance analysis

## 3. Preprocessing + Encoding

### 3.1. Lý thuyết tiền xử lý dữ liệu
- Mục tiêu của preprocessing là làm dữ liệu sạch, đồng nhất thang đo và phù hợp với thuật toán phân loại.
- Quy tắc quan trọng để tránh data leakage: chỉ fit các bước xử lý trên tập train, sau đó transform cho validation/test.
- Missing values làm sai lệch thống kê và giảm độ ổn định khi huấn luyện, nên cần impute trước khi huấn luyện mô hình.

### 3.2. Lý thuyết Encoding
- Với biến phân loại, mô hình học máy cần đầu vào dạng số nên phải mã hóa (encoding).
- Cách phổ biến là One-Hot Encoding: mỗi giá trị hạng mục được tách thành một cột nhị phân 0/1.
- One-Hot tránh giả định thứ tự giữa các nhãn, phù hợp cho Logistic Regression, SVM, MLP và nhiều mô hình tuyến tính.
- Sau khi encode, cần đồng bộ cột giữa train/val/test theo cấu trúc của train để giữ tính nhất quán đặc trưng.

### 3.3. Chuẩn hóa đặc trưng số
- Với các biến số liên tục, chuẩn hóa bằng StandardScaler giúp tối ưu hội tụ nhanh và ổn định hơn.
- Công thức chuẩn hóa:

$$z = \frac{x-\mu}{\sigma}$$

- Tương tự các bước khác, scaler chỉ được fit trên train rồi áp dụng cho val/test.

In [4]:
# 3) Preprocessing + encoding (fit on train only)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Keep raw copies for later comparison
X_train_raw = X_train.copy()
X_val_raw = X_val.copy()
X_test_raw = X_test.copy()

# Detect original column groups
cat_cols = X_train_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X_train_raw.columns if c not in cat_cols]

# Missing stats BEFORE preprocessing
missing_before = pd.DataFrame({
    "train_missing": X_train_raw.isna().sum(),
    "val_missing": X_val_raw.isna().sum(),
    "test_missing": X_test_raw.isna().sum(),
})
missing_before["total_missing"] = missing_before.sum(axis=1)

# 1) Imputation
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_imp = X_train_raw.copy()
X_val_imp = X_val_raw.copy()
X_test_imp = X_test_raw.copy()

if num_cols:
    X_train_imp[num_cols] = num_imputer.fit_transform(X_train_raw[num_cols])
    X_val_imp[num_cols] = num_imputer.transform(X_val_raw[num_cols])
    X_test_imp[num_cols] = num_imputer.transform(X_test_raw[num_cols])

if cat_cols:
    X_train_imp[cat_cols] = cat_imputer.fit_transform(X_train_raw[cat_cols])
    X_val_imp[cat_cols] = cat_imputer.transform(X_val_raw[cat_cols])
    X_test_imp[cat_cols] = cat_imputer.transform(X_test_raw[cat_cols])

# Missing stats AFTER imputation
missing_after = pd.DataFrame({
    "train_missing": X_train_imp.isna().sum(),
    "val_missing": X_val_imp.isna().sum(),
    "test_missing": X_test_imp.isna().sum(),
})
missing_after["total_missing"] = missing_after.sum(axis=1)

# 2) One-hot encoding for categorical columns with train-defined schema
if cat_cols:
    for col in cat_cols:
        train_categories = pd.Categorical(X_train_imp[col]).categories
        X_train_imp[col] = pd.Categorical(X_train_imp[col], categories=train_categories)
        X_val_imp[col] = pd.Categorical(X_val_imp[col], categories=train_categories)
        X_test_imp[col] = pd.Categorical(X_test_imp[col], categories=train_categories)

    X_train_enc = pd.get_dummies(X_train_imp, columns=cat_cols, drop_first=False)
    X_val_enc = pd.get_dummies(X_val_imp, columns=cat_cols, drop_first=False)
    X_test_enc = pd.get_dummies(X_test_imp, columns=cat_cols, drop_first=False)

    X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)
else:
    X_train_enc = X_train_imp.copy()
    X_val_enc = X_val_imp.copy()
    X_test_enc = X_test_imp.copy()

# 3) Scale only original numerical columns
scaler = StandardScaler()
X_train_prep = X_train_enc.copy()
X_val_prep = X_val_enc.copy()
X_test_prep = X_test_enc.copy()

scale_cols = [c for c in num_cols if c in X_train_prep.columns]
if scale_cols:
    X_train_prep[scale_cols] = scaler.fit_transform(X_train_prep[scale_cols])
    X_val_prep[scale_cols] = scaler.transform(X_val_prep[scale_cols])
    X_test_prep[scale_cols] = scaler.transform(X_test_prep[scale_cols])

# Expose preprocessed feature sets for next sections
X_train, X_val, X_test = X_train_prep, X_val_prep, X_test_prep

print("=== Preprocessing + Encoding summary ===")
print(f"Original shape (train/val/test): {X_train_raw.shape} | {X_val_raw.shape} | {X_test_raw.shape}")
print(f"Categorical columns: {len(cat_cols)}")
print(f"Numerical columns: {len(num_cols)}")
print(f"Total missing BEFORE: {int(missing_before['total_missing'].sum())}")
print(f"Total missing AFTER : {int(missing_after['total_missing'].sum())}")
print(f"Encoded shape (train/val/test): {X_train.shape} | {X_val.shape} | {X_test.shape}")

# Quick quality checks
display(missing_before.sort_values("total_missing", ascending=False).head(10))
display(missing_after.sort_values("total_missing", ascending=False).head(10))

if scale_cols:
    scaled_stats = pd.DataFrame({
        "mean_after_scaling": X_train[scale_cols].mean(),
        "std_after_scaling": X_train[scale_cols].std(ddof=0),
    })
    display(scaled_stats.head(10))

=== Preprocessing + Encoding summary ===
Original shape (train/val/test): (406708, 54) | (58101, 54) | (116203, 54)
Categorical columns: 0
Numerical columns: 54
Total missing BEFORE: 0
Total missing AFTER : 0
Encoded shape (train/val/test): (406708, 54) | (58101, 54) | (116203, 54)


,train_missing,val_missing,test_missing,total_missing
Elevation,0,0,0,0
Aspect,0,0,0,0
Slope,0,0,0,0
Horizontal_Distance_To_Hydrology,0,0,0,0
Vertical_Distance_To_Hydrology,0,0,0,0
Horizontal_Distance_To_Roadways,0,0,0,0
Hillshade_9am,0,0,0,0
Hillshade_Noon,0,0,0,0
Hillshade_3pm,0,0,0,0
Horizontal_Distance_To_Fire_Points,0,0,0,0


,train_missing,val_missing,test_missing,total_missing
Elevation,0,0,0,0
Aspect,0,0,0,0
Slope,0,0,0,0
Horizontal_Distance_To_Hydrology,0,0,0,0
Vertical_Distance_To_Hydrology,0,0,0,0
Horizontal_Distance_To_Roadways,0,0,0,0
Hillshade_9am,0,0,0,0
Hillshade_Noon,0,0,0,0
Hillshade_3pm,0,0,0,0
Horizontal_Distance_To_Fire_Points,0,0,0,0


,mean_after_scaling,std_after_scaling
Elevation,-4.367647e-19,1.0
Aspect,1.799470e-18,1.0
Slope,1.086670e-17,1.0
Horizontal_Distance_To_Hydrology,4.891764e-18,1.0
Vertical_Distance_To_Hydrology,-1.593318e-17,1.0
Horizontal_Distance_To_Roadways,1.380176e-18,1.0
Hillshade_9am,3.287964e-17,1.0
Hillshade_Noon,1.181012e-17,1.0
Hillshade_3pm,-1.422106e-17,1.0
Horizontal_Distance_To_Fire_Points,1.432588e-18,1.0


### 3.4. Phân tích kết quả tiền xử lý và mã hóa

- Kết quả chạy cho thấy Total missing BEFORE = 0 và Total missing AFTER = 0, nên dữ liệu hiện tại không cần bù thiếu nhưng pipeline imputation vẫn sẵn sàng cho trường hợp dữ liệu mới có missing.
- Số cột categorical phát hiện theo dtype là 0, vì vậy bước One-Hot Encoding không làm tăng số chiều; shape train/val/test giữ nguyên (54 đặc trưng).
- Dù không phát sinh cột mã hóa mới ở bộ dữ liệu này, logic đồng bộ schema train-val-test vẫn quan trọng để tránh mismatch khi làm việc với dữ liệu có biến phân loại.
- Bảng scaled_stats cho thấy mean các biến số xấp xỉ 0 và std xấp xỉ 1 trên train, xác nhận StandardScaler hoạt động đúng.
- Toàn bộ bước fit (imputer, scaler) đều thực hiện trên train rồi mới transform cho val/test, nên quy trình tránh được data leakage và đảm bảo đánh giá mô hình đáng tin cậy.

## 4. Logistic Regression
- GD (from scratch)
- Newton/IRLS (from scratch)
- binary + multiclass

In [4]:
# 4) Logistic Regression (from scratch)
# TODO: GD solver
# TODO: Newton/IRLS solver
# TODO: binary and multiclass strategy

## 5. LDA / QDA
- train and visualize decision boundary

In [5]:
# 5) LDA / QDA
# TODO: implement training + decision boundary visualization

## 6. Perceptron
- training dynamics and convergence behavior

In [6]:
# 6) Perceptron
# TODO: implement and analyze convergence behavior

## 7. Regularization
- L1 vs L2
- class-weighted loss

In [7]:
# 7) Regularization
# TODO: L1 vs L2 comparison
# TODO: class-weighted loss

## 8. Evaluation
- accuracy, precision, recall, F1
- confusion matrix
- ROC + AUC
- PR curve
- 5-fold CV

In [8]:
# 8) Evaluation
# TODO: accuracy, precision, recall, F1
# TODO: confusion matrix
# TODO: ROC/AUC and PR curve
# TODO: 5-fold CV summary

## 9. Statistical Test
- McNemar test

In [9]:
# 9) Statistical test
# TODO: McNemar test

## 10. Analysis
- model comparison
- error analysis

In [10]:
# 10) Analysis
# TODO: model comparison and error analysis